# RMSD of AFQMC, CCSD(T), and CCSDT vs CCSDT(Q)

Reference method is **CCSDT(Q)**. We compute the root-mean-square deviation of
AFQMC, CCSD(T), and CCSDT total energies relative to CCSDT(Q), over the
molecules/atoms for which both the predicted value and a CCSDT(Q) value are available.

Reference-file column mapping:
- `E_UHF-UCCSD(T) (Eh)`  -> CCSD(T)
- `E_UHF-UCCSDT (Eh)`    -> CCSDT
- `E_CCSDT(Q) (Eh)`      -> CCSDT(Q)  (reference)

Uses numpy only (env: `pyscf_afqmc`).

In [1]:
import numpy as np

HARTREE2KCAL = 627.5094740631


def to_float(tok):
    """Parse a token to float; 'missing'/non-numeric -> NaN."""
    try:
        return float(tok)
    except ValueError:
        return np.nan

In [2]:
# --- Load AFQMC energies: Molecule  Shell  E_AFQMC  E_AFQMC_err ---
afqmc = {}
with open('afqmc_vtzfp.dat') as f:
    for line in f.readlines()[2:]:
        p = line.split()
        if len(p) < 4 or p[0].startswith('#'):
            continue
        afqmc[p[0]] = to_float(p[2])
print('AFQMC entries:', len(afqmc))

AFQMC entries: 200


In [3]:
# --- Load reference CC energies ---
# cols: Molecule  E_HF  E_CCSD  E_CCSD(T)  E_CCSDT  E_CCSDT(Q)
ref = {}  # name -> dict of method energies
with open('reference_vtzfp.dat') as f:
    for line in f.readlines()[2:]:
        p = line.split()
        if len(p) < 6 or p[0].startswith('#'):
            continue
        ref[p[0]] = {
            'CCSD(T)': to_float(p[3]),
            'CCSDT': to_float(p[4]),
            'CCSDT(Q)': to_float(p[5]),
        }
print('reference entries:', len(ref))
print('missing CCSDT(Q):',
      [n for n, d in ref.items() if not np.isfinite(d['CCSDT(Q)'])])

reference entries: 211
missing CCSDT(Q): ['c2cl6', 'c2f6']


In [4]:
# --- Names present in each file (informational) ---
only_ref = sorted(set(ref) - set(afqmc))
only_afqmc = sorted(set(afqmc) - set(ref))
print('in reference but not AFQMC:', only_ref)
print('in AFQMC but not reference:', only_afqmc)

in reference but not AFQMC: ['al', 'b', 'c', 'cl', 'f', 'h', 'n', 'o', 'p', 's', 'si']
in AFQMC but not reference: []


In [5]:
# --- Compute RMSD vs CCSDT(Q) ---
def rmsd_vs_ref(get_pred):
    """RMSD of predicted vs CCSDT(Q) over molecules where both are finite."""
    diffs = []
    for name, d in ref.items():
        eq = d['CCSDT(Q)']
        pred = get_pred(name, d)
        if pred is None or not (np.isfinite(eq) and np.isfinite(pred)):
            continue
        diffs.append(pred - eq)
    diffs = np.array(diffs)
    return np.sqrt(np.mean(diffs**2)), len(diffs)


methods = {
    'AFQMC': lambda name, d: afqmc.get(name),
    'CCSD(T)': lambda name, d: d['CCSD(T)'],
    'CCSDT': lambda name, d: d['CCSDT'],
}

print('RMSD vs CCSDT(Q) (reference)')
print('=' * 60)
for label, fn in methods.items():
    r_eh, n = rmsd_vs_ref(fn)
    print(f'{label:>8s}  (N={n:3d})   '
          f'{r_eh*1e3:8.4f} mEh   {r_eh*HARTREE2KCAL:8.4f} kcal/mol')

RMSD vs CCSDT(Q) (reference)
   AFQMC  (N=198)     1.4378 mEh     0.9023 kcal/mol
 CCSD(T)  (N=209)     2.3950 mEh     1.5029 kcal/mol
   CCSDT  (N=209)     2.2440 mEh     1.4081 kcal/mol
